## **05. Set E — ABSA BERT Sentiment Scoring**

This notebook applies `yangheng/deberta-v3-base-absa-v1.1` to extract aspect-level sentiment scores for five service dimensions:

| Aspect | Label passed to model |
|---|---|
| seat | `seat comfort and legroom` |
| food | `food and beverage quality` |
| staff | `cabin crew and customer service` |
| ground_service | `flight delay, baggage, and check-in` |
| entertainment | `inflight entertainment and wifi` |

**Output:** `05_absa_scores.csv` — original dataframe + 5 new `absa_*` columns (weighted score: P_pos − P_neg, range −1 to +1)

> **Note:** This model was trained on restaurant/laptop/retail reviews (SemEval-2014, MAMS, etc.), not airline reviews. Domain mismatch is an acknowledged limitation, particularly for `ground_service`. See final report for discussion.

## **1. Environment Setup**

In [ ]:
# Install dependencies (Colab)
!pip install transformers torch -q

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## **2. Mount Google Drive & Load Data**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# ── Update this path to match your Google Drive structure ──
DATA_PATH = '/content/drive/MyDrive/airline-review-sentiment-analysis/1_data/processed/04_aspect_scores.csv'
OUT_PATH  = '/content/drive/MyDrive/airline-review-sentiment-analysis/1_data/processed/05_absa_scores.csv'

DATA_PATH = '/content/drive/MyDrive/Airline-Review-Sentiment-Classifier/1_data/processed/04_aspect_scores.csv'
OUT_PATH  = '/content/drive/MyDrive/Airline-Review-Sentiment-Classifier/1_data/processed/05_absa_scores.csv'

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape}")
df.head(2)

## **3. Load ABSA Model**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

MODEL_NAME = 'yangheng/deberta-v3-base-absa-v1.1'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# top_k=None returns all class probabilities (Negative / Neutral / Positive)
absa_classifier = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    top_k=None,
    device=0 if torch.cuda.is_available() else -1
)

print("Model loaded.")

## **4. Define Aspect Labels**

In [ ]:
# Aspect labels passed to the model as text_pair
# Expanded from single-word keys to fuller phrases to mitigate domain mismatch
ASPECT_LABELS = {
    'seat'            : 'seat comfort and legroom',
    'food'            : 'food and beverage quality',
    'staff'           : 'cabin crew and customer service',
    'ground_service'  : 'flight delay, baggage, and check-in',
    'entertainment'   : 'inflight entertainment and wifi'
}

## **5. Define Scoring Function**

Model output per aspect: `[{label: 'Negative', score: P_neg}, {label: 'Neutral', score: P_neu}, {label: 'Positive', score: P_pos}]`

Weighted score = `P_pos − P_neg` → range −1 to +1, directly comparable to Set C VADER compound scores.

In [ ]:
def get_weighted_score(probs: list) -> float:
    """Convert probability list to weighted sentiment score (P_pos - P_neg)."""
    prob_dict = {item['label']: item['score'] for item in probs}
    return round(prob_dict.get('Positive', 0) - prob_dict.get('Negative', 0), 4)


def score_aspects(text: str) -> dict:
    """Return weighted ABSA score for each aspect given a review text."""
    if not isinstance(text, str) or text.strip() == '':
        return {f'absa_{asp}': None for asp in ASPECT_LABELS}

    # Truncate to 512 tokens (DeBERTa limit) — pipeline handles this but explicit is safer
    text = text[:2000]

    results = {}
    for asp, label in ASPECT_LABELS.items():
        try:
            probs = absa_classifier(text, text_pair=label)[0]
            results[f'absa_{asp}'] = get_weighted_score(probs)
        except Exception as e:
            results[f'absa_{asp}'] = None
    return results


# ── Quick sanity check ──
sample = df['cleaned_review_BE'].iloc[0]
print("Sample review:", sample[:120], "...")
print("\nABSA scores:")
print(score_aspects(sample))

## **6. Run Inference on Full Dataset**

> ⏱️ Expected time: ~30–60 min for 22,980 rows on Colab T4 GPU.  
> Progress is saved to Drive every 1,000 rows so you can resume if the session drops.

In [ ]:
import os
from tqdm.auto import tqdm

CHECKPOINT_PATH = OUT_PATH.replace('.csv', '_checkpoint.csv')
SAVE_EVERY = 1000

# ── Resume from checkpoint if it exists ──
if os.path.exists(CHECKPOINT_PATH):
    df_done = pd.read_csv(CHECKPOINT_PATH)
    start_idx = len(df_done)
    print(f"Resuming from row {start_idx}")
else:
    df_done = df.copy()
    for col in [f'absa_{asp}' for asp in ASPECT_LABELS]:
        df_done[col] = None
    start_idx = 0
    print("Starting from scratch.")

# ── Inference loop ──
texts = df['cleaned_review_BE'].tolist()

for i in tqdm(range(start_idx, len(df)), desc='ABSA scoring'):
    scores = score_aspects(texts[i])
    for col, val in scores.items():
        df_done.at[i, col] = val

    if (i + 1) % SAVE_EVERY == 0:
        df_done.to_csv(CHECKPOINT_PATH, index=False)
        print(f"  Checkpoint saved at row {i+1}")

print("\nInference complete.")

## **7. Save Final Output**

In [ ]:
df_done.to_csv(OUT_PATH, index=False)
print(f"Saved → {OUT_PATH}")
print(f"Final shape: {df_done.shape}")
df_done.info()

## **8. Quick Validation**

In [ ]:
absa_cols = [f'absa_{asp}' for asp in ASPECT_LABELS]

print("=== Score Distribution ===")
print(df_done[absa_cols].describe().round(3))

print("\n=== Non-null counts ===")
print(df_done[absa_cols].notna().sum())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for ax, col in zip(axes, absa_cols):
    df_done[col].dropna().hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col.replace('absa_', ''))
    ax.set_xlabel('Score (P_pos - P_neg)')
    ax.axvline(0, color='red', linestyle='--', linewidth=0.8)
plt.suptitle('Set E — ABSA Score Distributions', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Compare Set C vs Set E on same aspects (correlation check)
compare_pairs = [
    ('aspect_seat',           'absa_seat'),
    ('aspect_food',           'absa_food'),
    ('aspect_staff',          'absa_staff'),
    ('aspect_ground_service', 'absa_ground_service'),
    ('aspect_entertainment',  'absa_entertainment'),
]

print("=== Pearson Correlation: Set C vs Set E ===")
for c_col, e_col in compare_pairs:
    corr = df_done[[c_col, e_col]].dropna().corr().iloc[0, 1]
    print(f"  {c_col:<30} vs {e_col:<25} r = {corr:.3f}")